# OULAD processing and feature extraction

This notebook documents the data layer of the Smart Study Planner. It uses the same reusable functions in `src/data/` and `src/features/` as the command-line pipeline, then exposes their cleaning decisions and validation results in notebook cells.

The notebook covers raw-table inspection, conservative cleaning, validation, enrollment-level feature extraction, and colorful descriptive plots.

## Notebook map

The workflow is intentionally ordered: inspect the source tables first, clean without inventing values, validate the cleaned populations, then create the modeling table. The cells below call project functions so that notebook results and command-line results use the same implementation.

## Data expectations

Place the seven OULAD CSV files in `data/raw/oulad/`. The required modeling tables are `studentInfo.csv`, `assessments.csv`, and `studentAssessment.csv`; registration and VLE tables are used when present.

In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'src/data/cleaner.py').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / 'data/raw/oulad'
PROCESSED_DIR = ROOT / 'data/processed/oulad'

from src.data.cleaner import clean_oulad_tables, write_clean_tables
from src.data.loader import load_clean_table, load_oulad_tables
from src.features.feature_engineering import build_oulad_features

print(f'Repository: {ROOT}')
print(f'Raw data: {RAW_DIR}')
print(f'Processed data: {PROCESSED_DIR}')

### What the OULAD tables represent

- `studentInfo`: one student-course enrollment and its final outcome.
- `studentRegistration`: registration and unregistration dates for an enrollment.
- `assessments`: definitions of assignments or exams offered by a course presentation.
- `studentAssessment`: one student's result record for one assessment.
- `vle`: online resource definitions.
- `studentVle`: student-resource-day activity records containing click counts.
- `courses`: course presentation metadata.

These tables have different row units. A row count must therefore be labeled with its table and must not be interpreted as a student count unless the table is `studentInfo` and the population is explicitly defined.

## 1. Inspect the raw tables

This first summary describes the unit represented by each table. These are different populations: an enrollment, assessment definition, assessment result, or activity record must not be treated as the same unit.

In [ ]:
raw_tables = load_oulad_tables(RAW_DIR)
raw_summary = pd.DataFrame(
    [
        {'table': name, 'rows': len(frame), 'columns': len(frame.columns),
         'memory_mb': frame.memory_usage(deep=True).sum() / 1024**2}
        for name, frame in raw_tables.items()
    ]
).sort_values('table')
display(raw_summary.reset_index(drop=True).style.format({'memory_mb': '{:.1f}'}))

In [ ]:
# Show the schema of each loaded table without printing large dataframes.
schema = pd.DataFrame(
    {name: pd.Series(frame.dtypes.astype(str).to_dict()) for name, frame in raw_tables.items()}
).fillna('—')
display(schema)

## 1.5 Raw quality audit

Before changing any values, this audit records duplicate complete rows, missing key fields, and overall cell missingness for each source table. A missing score is an unknown value, not a zero; it is measured separately after cleaning. These checks are diagnostics, not reasons to delete rows automatically.

In [ ]:
key_columns = {
    'studentInfo': ['id_student', 'code_presentation'],
    'studentRegistration': ['id_student', 'code_presentation'],
    'assessments': ['id_assessment', 'code_presentation'],
    'studentAssessment': ['id_student', 'id_assessment'],
    'vle': ['id_site', 'code_presentation'],
    'studentVle': ['id_student', 'id_site', 'date'],
}

raw_quality_rows = []
for name, frame in raw_tables.items():
    keys = [column for column in key_columns.get(name, []) if column in frame.columns]
    missing_key_rows = int(frame[keys].isna().any(axis=1).sum()) if keys else 0
    raw_quality_rows.append({
        'table': name,
        'rows': len(frame),
        'duplicate_complete_rows': int(frame.duplicated().sum()),
        'rows_missing_key_fields': missing_key_rows,
        'missing_cells_percent': round(float(frame.isna().mean().mean() * 100), 3),
    })

raw_quality = pd.DataFrame(raw_quality_rows).sort_values('table')
display(raw_quality.reset_index(drop=True))

### Cleaning decisions

The processing code uses validation and type normalization rather than aggressive imputation. Required columns and files cause an explicit error when absent. Scores outside 0–100 are treated as unknown, and existing missing scores remain unknown. No missing score is changed to zero.

For VLE data, `sum_click` is retained as an activity count. It is not converted to hours because OULAD does not provide a reliable duration ground truth.

## 2. Clean and validate

The cleaner checks required columns, converts numeric fields, keeps missing scores missing, converts out-of-range scores to missing, and removes only records that cannot be joined or validated. It does not invent study time or performance values.

In [ ]:
cleaned_tables = clean_oulad_tables(raw_tables)
row_reconciliation = pd.DataFrame(
    [
        {'table': name, 'raw_rows': len(raw_tables[name]), 'clean_rows': len(frame),
         'removed_rows': len(raw_tables[name]) - len(frame)}
        for name, frame in cleaned_tables.items()
    ]
).sort_values('table')
display(row_reconciliation.reset_index(drop=True))

# Persisting is optional in the notebook; this is the same operation used by the CLI.
write_clean_tables(
    cleaned_tables,
    PROCESSED_DIR,
    raw_row_counts={name: len(frame) for name, frame in raw_tables.items()},
)
print(f'Cleaned tables written to {PROCESSED_DIR}')

### Cleaning validation checks

These checks make the cleaning contract visible. Identifier rows removed by the cleaner must be gone; assessment scores must be either missing or within 0–100; assessment and VLE definition identifiers must be unique; and invalid final outcomes must be represented as missing rather than relabeled.

In [ ]:
validation_rows = [
    {
        'table': 'studentAssessment',
        'check': 'missing required identifiers',
        'violations': int(cleaned_tables['studentAssessment'][['id_student', 'id_assessment']].isna().any(axis=1).sum()),
    },
    {
        'table': 'studentAssessment',
        'check': 'scores outside 0-100',
        'violations': int((cleaned_tables['studentAssessment']['score'].notna() & ~cleaned_tables['studentAssessment']['score'].between(0, 100)).sum()),
    },
    {
        'table': 'studentAssessment',
        'check': 'missing scores retained as unknown',
        'violations': int(cleaned_tables['studentAssessment']['score'].isna().sum()),
    },
    {
        'table': 'assessments',
        'check': 'missing assessment identifiers',
        'violations': int(cleaned_tables['assessments']['id_assessment'].isna().sum()),
    },
    {
        'table': 'assessments',
        'check': 'duplicate assessment identifiers',
        'violations': int(cleaned_tables['assessments']['id_assessment'].duplicated().sum()),
    },
    {
        'table': 'vle',
        'check': 'duplicate VLE site identifiers',
        'violations': int(cleaned_tables['vle']['id_site'].duplicated().sum()),
    },
    {
        'table': 'studentVle',
        'check': 'missing activity identifiers/clicks',
        'violations': int(cleaned_tables['studentVle'][['id_student', 'id_site', 'sum_click']].isna().any(axis=1).sum()),
    },
    {
        'table': 'studentInfo',
        'check': 'invalid non-null final outcomes',
        'violations': int((cleaned_tables['studentInfo']['final_result'].notna() & ~cleaned_tables['studentInfo']['final_result'].isin(['Distinction', 'Pass', 'Fail', 'Withdrawn'])).sum()),
    },
]
validation = pd.DataFrame(validation_rows)
display(validation)
print('All zero-violation checks passed; missing scores are intentionally reported as expected unknowns.')
assert (validation.loc[validation['check'] != 'missing scores retained as unknown', 'violations'] == 0).all()

In [ ]:
# Verify that the persisted CSVs can be reloaded and match the cleaned frames.
round_trip_rows = []
for name, frame in cleaned_tables.items():
    persisted = load_clean_table(name, PROCESSED_DIR)
    rows_match = len(persisted) == len(frame)
    columns_match = list(persisted.columns) == list(frame.columns)
    assert rows_match and columns_match
    round_trip_rows.append({'table': name, 'rows_match': rows_match, 'columns_match': columns_match})

manifest = json.loads((PROCESSED_DIR / 'manifest.json').read_text(encoding='utf-8'))
assert set(manifest['tables']) == set(cleaned_tables)
round_trip = pd.DataFrame(round_trip_rows).sort_values('table')
display(round_trip.reset_index(drop=True))
print('Persisted cleaned tables and manifest reload successfully.')

In [ ]:
# Colorful view of rows removed by validation or deduplication.
fig, ax = plt.subplots(figsize=(10, 5))
plot_data = row_reconciliation.sort_values('removed_rows')
ax.barh(plot_data['table'], plot_data['removed_rows'], color=plt.cm.magma(
    range(50, 50 + 30 * len(plot_data), 30)
))
ax.set_title('Cleaning impact by table', loc='left', fontsize=15, weight='bold')
ax.set_xlabel('Rows removed during deterministic cleaning')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
student_info = cleaned_tables['studentInfo']
student_assessment = cleaned_tables['studentAssessment']
quality = pd.DataFrame({
    'measure': ['known scores', 'missing scores', 'minimum score', 'maximum score'],
    'value': [
        int(student_assessment['score'].notna().sum()),
        int(student_assessment['score'].isna().sum()),
        student_assessment['score'].min(),
        student_assessment['score'].max(),
    ],
})
display(quality)
display(student_info['final_result'].value_counts(dropna=False).rename_axis('final_result').to_frame('enrollments'))

In [ ]:
# Data-quality plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

outcomes = student_info['final_result'].value_counts().reindex(['Distinction', 'Pass', 'Fail', 'Withdrawn'])
axes[0].bar(outcomes.index, outcomes.values, color=['#2ca25f', '#66c2a4', '#fc8d62', '#e78ac3'])
axes[0].set_title('Enrollment outcomes', loc='left', weight='bold')
axes[0].set_ylabel('Enrollments')

missing = student_assessment.isna().mean().sort_values(ascending=False).head(8) * 100
axes[1].barh(missing.index[::-1], missing.values[::-1], color=plt.cm.viridis_r(range(40, 40 + 20 * len(missing), 20)))
axes[1].set_title('Missingness in assessment table', loc='left', weight='bold')
axes[1].set_xlabel('Missing share (%)')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
plt.show()

### How to read the quality plots

The outcome chart shows enrollments by final result; it is not a unique-student chart. The missingness chart shows unknown information in the assessment table. Missingness is a data-quality fact, not evidence of no activity or no effort. The charts are descriptive and do not establish causal relationships.

## 3. Extract enrollment-level features

Assessment results are joined to assessment definitions before aggregation. Activity is aggregated by student and course presentation. The output contains one labeled enrollment row and is the input to the model comparison notebook.

In [ ]:
features = build_oulad_features(cleaned_tables, clean=False)
print(f'Feature matrix: {features.shape[0]:,} rows × {features.shape[1]:,} columns')
display(features.head())
display(features.dtypes.astype(str).rename('dtype').to_frame())

In [ ]:
# Feature distributions and risk-group comparison
numeric_features = [
    column for column in ['assessment_score_mean', 'assessment_count', 'vle_clicks', 'vle_active_days']
    if column in features.columns
]
fig, axes = plt.subplots(1, len(numeric_features), figsize=(5 * len(numeric_features), 4.5))
if len(numeric_features) == 1:
    axes = [axes]
for ax, column in zip(axes, numeric_features):
    groups = [features.loc[features['target_risk'] == label, column].dropna() for label in [0, 1]]
    ax.boxplot(groups, tick_labels=['lower risk', 'higher risk'], patch_artist=True,
               boxprops={'facecolor': '#9ecae1'}, medianprops={'color': '#d95f0e', 'linewidth': 2})
    ax.set_title(column.replace('_', ' ').title(), loc='left', weight='bold')
    ax.grid(axis='y', alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

### Feature definitions

Assessment features summarize the available assessment records for one enrollment: number of assessments, score mean, score spread, score range, and count of missing scores. Registration fields preserve timing information supplied by OULAD. VLE features summarize clicks, active days, and distinct resources.

The final outcome is converted into `target_risk` for supervised learning. Identifiers and the raw final outcome are metadata and are excluded from model inputs. These full-course aggregates are useful for a baseline but may leak information for an early-warning use case.

## Interpretation limits

These plots describe associations in OULAD. They do not prove that activity caused an outcome, that clicks equal study hours, or that a particular amount of study is optimal. The next modeling stage must define a prediction cutoff to prevent full-course information from leaking into an early-warning model.